# Calorvia · Carnet de travail Text-to-SQL

## Journal PostgreSQL

Le test guidé sur le Z4 a confirmé quatre refus d’accès (0,23 s). La première question s’est ensuite arrêtée à l’initialisation du journal via Docker, avant Qwen et MCP.

Exécuter la cellule suivante depuis le dépôt, avec son noyau Python habituel. Elle reproduit l’initialisation du schéma `text2sql_journal` dans une transaction annulée et affiche le diagnostic natif. Aucun modèle n’est appelé.

In [ ]:
import os
import subprocess
from pathlib import Path

# Le carnet peut être ouvert depuis text2sql/ ou data/text2sql/.
cwd = Path.cwd().resolve()
racine = next((p for p in (cwd, *cwd.parents) if (p / "text2sql/journal.sql").is_file()), None)
if racine is None:
    raise RuntimeError("Ouvrir ce carnet depuis le dépôt calorvia_rag.")

from dotenv import load_dotenv
for fichier in (racine / ".env", racine / "text2sql/.env"):
    load_dotenv(fichier, override=False)

if os.environ.get("T2SQL_JOURNAL_DSN"):
    raise RuntimeError("Un DSN est maintenant configuré. Le dernier échec utilisait Docker. Vérifier le transport avant de poursuivre.")

conteneur = os.environ.get("T2SQL_JOURNAL_CONTAINER", "helpdesk-ops-postgres-1")
ddl = (racine / "text2sql/journal.sql").read_text(encoding="utf-8")
sql = """BEGIN;
SELECT current_database() AS base, current_user AS compte,
       current_setting('transaction_read_only') AS lecture_seule,
       has_database_privilege(current_database(), 'CREATE') AS peut_creer_schema,
       to_regnamespace('text2sql_journal') IS NOT NULL AS schema_existe;
""" + ddl + "\nROLLBACK;\n"
test = subprocess.run(
    ["docker", "exec", "-i", conteneur, "psql", "-X", "-U", "calorvia_app",
     "-d", "calorvia_rag", "-v", "ON_ERROR_STOP=1", "-f", "-"],
    input=sql, text=True, capture_output=True, timeout=20,
)
print(test.stdout.strip())
if test.stderr.strip():
    print(test.stderr.strip())
print("Code retour :", test.returncode)
if test.returncode == 0:
    print("Initialisation testée, transaction annulée.")

**Interprétation.** Un code non nul et le message de Docker ou PostgreSQL indiquent où reprendre. Une erreur SQL interrompt `psql` et la fermeture de connexion annule la transaction. Un code zéro avec `ROLLBACK` confirme que le compte peut exécuter le script dans les conditions actuelles. L’initialisation persistante et la première question Qwen via MCP constituent l’étape suivante.